# 🧪 Resume Genie Agent — Testing Notebook

## Purpose
Interactively test every routing path, tool chain, and edge case of the LangGraph agent.

## Test Categories
1. **Single Tool** — Resume Checker, Resume Scorer, Cover Letter, Career Coach
2. **Multi-Tool Chaining** — Full analysis (checker + scorer + cover letter)
3. **Edge Cases** — Missing JD, conversation memory, ambiguous queries

---
## 🔧 Part 1: Environment Setup

In [ ]:
# ============================================================================
# OPTIONAL: Install Required Packages
# ============================================================================
# !pip install -qq langchain_groq langchain_community langchain_core langgraph pypdf python-dotenv

In [1]:
# ============================================================================
# ENVIRONMENT SETUP
# ============================================================================
import os
import sys
import time
from dotenv import load_dotenv

# Add project root to path so agent package is importable
sys.path.insert(0, os.path.abspath(".."))

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, "GROQ_API_KEY not found in .env"

print("✅ Environment loaded")

✅ Environment loaded


In [2]:
# ============================================================================
# INITIALIZE LLM + GRAPH
# ============================================================================
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage
from langchain_community.document_loaders import PyPDFLoader
from agent.graph import build_graph, TOOL_LABELS

llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY)
graph = build_graph(llm)

print(f"✅ Graph compiled with nodes: {list(graph.get_graph().nodes.keys())}")

/Users/sourav.banerjee/Documents/Codebases/AI_Engineering/Resume-Genie/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Graph compiled with nodes: ['__start__', 'router', 'cover_letter', 'resume_scorer', 'resume_checker', 'career_coach', 'responder', '__end__']


In [3]:
# ============================================================================
# LOAD SAMPLE RESUME
# ============================================================================
loader = PyPDFLoader("../content/Resume.pdf")
documents = loader.load()
resume_text = "\n\n".join(doc.page_content for doc in documents)

print(f"📄 Loaded {len(documents)} page(s), {len(resume_text)} chars")

📄 Loaded 5 page(s), 11364 chars


In [4]:
# ============================================================================
# SAMPLE JOB DESCRIPTION
# ============================================================================
job_description = """
What You'll Bring:

Bachelor's or higher in a quantitative field (e.g., statistics, mathematics,
computer science, engineering) with a strong academic record.
8 years or above experience in supporting clients and at least 4 years in
analytics, ideally in industries like financial services, insurance, or
digital marketing.
Proven success in end-to-end model development for at least 4 years.
Strong analytical, critical thinking, and problem-solving skills with a
proactive mindset.
Advanced programming skills and aptitude: proficiency with statistical
programs such as R or Python; experience with other programming languages
(SQL, C/C++, Java, Ab Initio) and platforms (Hive, Spark)
Capable of leading complex analytics projects with minimal supervision and
cross-functional collaboration.
Excellent project/time management; able to handle multiple initiatives in a
fast-paced environment.
Strong business acumen and communication skills; able to influence
stakeholders at all levels.
Deep understanding of industry trends and customer needs to identify business
opportunities.
Skilled in translating technical insights into actionable business
recommendations.
Occasional travel required.
"""

print("📋 Job description loaded")

📋 Job description loaded


In [5]:
# ============================================================================
# HELPER: Run a query through the graph and print results
# ============================================================================
def run_query(query, resume=resume_text, jd=None, history=None):
    """Run a query through the agent graph and print node-by-node execution."""
    messages = list(history) if history else []
    messages.append(HumanMessage(content=query))

    state = {
        "messages": messages,
        "resume_text": resume,
        "job_description": jd,
        "tool_choice": None,
        "tool_chain": None,
        "tool_outputs": None,
    }

    print(f"\n{'='*70}")
    print(f"QUERY: {query}")
    print(f"JD: {'provided' if jd else 'NOT provided'}")
    print(f"{'='*70}")

    start = time.time()
    ai_message = None
    nodes_visited = []

    for event in graph.stream(state, stream_mode="updates"):
        for node_name, node_output in event.items():
            elapsed = time.time() - start
            nodes_visited.append(node_name)

            # Print routing decision from router
            if node_name == "router":
                choice = node_output.get("tool_choice", "?")
                chain = node_output.get("tool_chain")
                chain_str = f" chain={chain}" if chain else ""
                print(f"  [{elapsed:5.1f}s] ROUTER → {choice}{chain_str}")
            elif node_name == "responder":
                print(f"  [{elapsed:5.1f}s] RESPONDER")
            else:
                label = TOOL_LABELS.get(node_name, node_name)
                print(f"  [{elapsed:5.1f}s] TOOL: {label}")

            # Capture AI message
            if "messages" in node_output:
                for msg in node_output["messages"]:
                    if isinstance(msg, AIMessage):
                        ai_message = msg

    total = time.time() - start
    print(f"\nRoute: {' → '.join(nodes_visited)}")
    print(f"Total time: {total:.1f}s")

    if ai_message:
        print(f"Response length: {len(ai_message.content)} chars")
        print(f"\n--- Response Preview (first 500 chars) ---")
        print(ai_message.content[:500])
        if len(ai_message.content) > 500:
            print("...")
    else:
        print("\n⚠️  No AI message returned")

    return ai_message, messages

print("✅ Helper function ready")

✅ Helper function ready


---
## 🔍 Part 2: Single Tool Tests — Resume Checker (no JD)

In [6]:
# ============================================================================
# TEST 1.1: Resume Checker — "Evaluate my resume"
# Expected: router → resume_checker → responder
# ============================================================================
msg, _ = run_query("Evaluate my resume")


QUERY: Evaluate my resume
JD: NOT provided


[resume_genie.agent] [ROUTER]    query="Evaluate my resume" | resume=uploaded | jd=NOT provided
[resume_genie.agent] [ROUTER]    decision=resume_checker (0.36s)
[resume_genie.agent] [ROUTER]    → single tool: resume_checker
[resume_genie.agent] [ROUTE]     after router → resume_checker
[resume_genie.agent] [NODE]      resume_checker started


  [  0.5s] ROUTER → resume_checker


[resume_genie.agent] [NODE]      resume_checker completed (3254 chars, 2.15s)
[resume_genie.agent] [ROUTE]     after tool → responder (single tool)
[resume_genie.agent] [RESPONDER] 1 tool output (resume_checker) → final message (3254 chars)


  [  2.7s] TOOL: Resume Evaluation
  [  2.7s] RESPONDER

Route: router → resume_checker → responder
Total time: 2.7s
Response length: 3254 chars

--- Response Preview (first 500 chars) ---
1. **Score**: 85 out of 100
2. **Strengths**:
   * The resume is comprehensive, covering a wide range of skills and experiences in Big Data Analytics, Cloud Computing, and Machine Learning.
   * It highlights a strong background in Azure, AWS, and other cloud platforms, as well as proficiency in various tools and technologies such as Hadoop, Spark, Kafka, and NoSQL databases.
   * The candidate has a proven track record of 8+ years of experience in software development, design, and execution of 
...


In [7]:
# ============================================================================
# TEST 1.2: Resume Checker — "Review my resume for quality"
# Expected: router → resume_checker → responder
# ============================================================================
msg, _ = run_query("Review my resume for quality")

[resume_genie.agent] [ROUTER]    query="Review my resume for quality" | resume=uploaded | jd=NOT provided



QUERY: Review my resume for quality
JD: NOT provided


[resume_genie.agent] [ROUTER]    decision=resume_checker (0.47s)
[resume_genie.agent] [ROUTER]    → single tool: resume_checker
[resume_genie.agent] [ROUTE]     after router → resume_checker
[resume_genie.agent] [NODE]      resume_checker started


  [  0.5s] ROUTER → resume_checker


[resume_genie.agent] [NODE]      resume_checker completed (2715 chars, 1.86s)
[resume_genie.agent] [ROUTE]     after tool → responder (single tool)
[resume_genie.agent] [RESPONDER] 1 tool output (resume_checker) → final message (2715 chars)


  [  2.4s] TOOL: Resume Evaluation
  [  2.4s] RESPONDER

Route: router → resume_checker → responder
Total time: 2.4s
Response length: 2715 chars

--- Response Preview (first 500 chars) ---
1. **Score**: 85/100
2. **Strengths**:
   - The resume clearly highlights the candidate's experience and skills in Big Data Analytics, Cloud Computing, and Machine Learning.
   - It provides a detailed overview of the candidate's technical skills, including proficiency in various tools and technologies such as Azure, AWS, Hadoop, Spark, and Python.
   - The candidate's experience in managing teams, leading projects, and collaborating with clients is well-documented, showcasing their ability to w
...


---
## 📊 Part 3: Single Tool Tests — Resume Scorer (with JD)

In [8]:
# ============================================================================
# TEST 2.1: Resume Scorer — "Score my resume against this JD"
# Expected: router → resume_scorer → responder
# ============================================================================
msg, _ = run_query("Score my resume against this job description", jd=job_description)

[resume_genie.agent] [ROUTER]    query="Score my resume against this job description" | resume=uploaded | jd=provided



QUERY: Score my resume against this job description
JD: provided


[resume_genie.agent] [ROUTER]    decision=resume_scorer (0.31s)
[resume_genie.agent] [ROUTER]    → single tool: resume_scorer
[resume_genie.agent] [ROUTE]     after router → resume_scorer
[resume_genie.agent] [NODE]      resume_scorer started


  [  0.3s] ROUTER → resume_scorer


[resume_genie.agent] [NODE]      resume_scorer completed (3010 chars, 2.08s)
[resume_genie.agent] [ROUTE]     after tool → responder (single tool)
[resume_genie.agent] [RESPONDER] 1 tool output (resume_scorer) → final message (3010 chars)


  [  2.4s] TOOL: ATS Match Analysis
  [  2.5s] RESPONDER

Route: router → resume_scorer → responder
Total time: 2.5s
Response length: 3010 chars

--- Response Preview (first 500 chars) ---
Score: 82/100
Overall Match: 80%

Keywords matched:
- Big Data Analytics
- Machine Learning
- Data Science
- Cloud Platforms (Azure, AWS)
- Programming languages (Python, Scala, R)
- Data Management
- Stakeholder Management
- Project Execution

Missing keywords:
- Quantitative field (statistics, mathematics, computer science) explicitly mentioned in education
- 8 years of experience in supporting clients
- End-to-end model development for at least 4 years
- Strong business acumen and communicati
...


In [9]:
# ============================================================================
# TEST 2.2: Resume Scorer — "How well does my resume match?"
# Expected: router → resume_scorer → responder
# ============================================================================
msg, _ = run_query("How well does my resume match this JD?", jd=job_description)

[resume_genie.agent] [ROUTER]    query="How well does my resume match this JD?" | resume=uploaded | jd=provided



QUERY: How well does my resume match this JD?
JD: provided


[resume_genie.agent] [ROUTER]    decision=resume_scorer (0.20s)
[resume_genie.agent] [ROUTER]    → single tool: resume_scorer
[resume_genie.agent] [ROUTE]     after router → resume_scorer
[resume_genie.agent] [NODE]      resume_scorer started


  [  0.2s] ROUTER → resume_scorer


[resume_genie.agent] [NODE]      resume_scorer completed (3245 chars, 2.27s)
[resume_genie.agent] [ROUTE]     after tool → responder (single tool)
[resume_genie.agent] [RESPONDER] 1 tool output (resume_scorer) → final message (3245 chars)


  [  2.5s] TOOL: ATS Match Analysis
  [  2.5s] RESPONDER

Route: router → resume_scorer → responder
Total time: 2.5s
Response length: 3245 chars

--- Response Preview (first 500 chars) ---
Score: 72/100
Overall Match: 80%

Keywords matched:
- Quantitative field (B.Tech in ECE)
- 8 years of experience in software development and data analytics
- Strong analytical and problem-solving skills
- Advanced programming skills (Python, Scala, R)
- Experience with Big Data Analytics, Cloud Platforms (Azure, AWS), and Machine Learning
- Strong business acumen and communication skills

Missing keywords:
- Bachelor's or higher in statistics, mathematics, or computer science (ECE is not a direc
...


---
## ✉️ Part 4: Single Tool Tests — Cover Letter (with JD)

In [10]:
# ============================================================================
# TEST 3.1: Cover Letter — "Write me a cover letter"
# Expected: router → cover_letter → responder
# ============================================================================
msg, _ = run_query("Write me a cover letter for this role", jd=job_description)

[resume_genie.agent] [ROUTER]    query="Write me a cover letter for this role" | resume=uploaded | jd=provided



QUERY: Write me a cover letter for this role
JD: provided


[resume_genie.agent] [ROUTER]    decision=cover_letter (0.15s)
[resume_genie.agent] [ROUTER]    → single tool: cover_letter
[resume_genie.agent] [ROUTE]     after router → cover_letter
[resume_genie.agent] [NODE]      cover_letter started


  [  0.2s] ROUTER → cover_letter


[resume_genie.agent] [NODE]      cover_letter completed (2354 chars, 1.78s)
[resume_genie.agent] [ROUTE]     after tool → responder (single tool)
[resume_genie.agent] [RESPONDER] 1 tool output (cover_letter) → final message (2354 chars)


  [  2.0s] TOOL: Cover Letter
  [  2.0s] RESPONDER

Route: router → cover_letter → responder
Total time: 2.0s
Response length: 2354 chars

--- Response Preview (first 500 chars) ---
March 18, 2024
Dear Hiring Manager,

I am excited to apply for the role that I came across on a job portal, which aligns with my skills and experience in Big Data Analytics and Cloud platforms. With a strong background in Big Data Analytics, ETL, Data Analytics, and Azure Cloud Platform, I am confident that I would be a strong fit for this position. My proven track record of 8+ years in diverse facets of software development, design, and execution of business applications, along with my experien
...


---
## 💬 Part 5: Single Tool Tests — Career Coach (no JD)

In [11]:
# ============================================================================
# TEST 4.1: Career Coach — Interview prep
# Expected: router → career_coach → END
# ============================================================================
msg, _ = run_query("How should I prepare for a data engineering interview?")

[resume_genie.agent] [ROUTER]    query="How should I prepare for a data engineering interview?" | resume=uploaded | jd=NOT provided
[resume_genie.agent] [ROUTER]    decision=career_coach (0.15s)
[resume_genie.agent] [ROUTER]    → single tool: career_coach
[resume_genie.agent] [ROUTE]     after router → career_coach



QUERY: How should I prepare for a data engineering interview?
JD: NOT provided


[resume_genie.agent] [NODE]      career_coach started


  [  0.2s] ROUTER → career_coach


[resume_genie.agent] [NODE]      career_coach completed (4060 chars, 2.57s)


  [  2.7s] TOOL: career_coach

Route: router → career_coach
Total time: 2.7s
Response length: 4060 chars

--- Response Preview (first 500 chars) ---
As a career coach and resume mentor, I'd be happy to help you prepare for a data engineering interview. Here's a comprehensive guide to get you started:

**Understand the Role**
--------------------

Data engineering is a broad field, and the responsibilities can vary depending on the company, team, and specific role. Familiarize yourself with the job description and requirements. Understand the skills and technologies mentioned, such as data processing, storage, and analytics.

**Key Areas to F
...


In [12]:
# ============================================================================
# TEST 4.2: Career Coach — Salary guidance
# Expected: router → career_coach → END
# ============================================================================
msg, _ = run_query("What salary should I expect for my next role?")

[resume_genie.agent] [ROUTER]    query="What salary should I expect for my next role?" | resume=uploaded | jd=NOT provided



QUERY: What salary should I expect for my next role?
JD: NOT provided


[resume_genie.agent] [ROUTER]    decision=career_coach (0.33s)
[resume_genie.agent] [ROUTER]    → single tool: career_coach
[resume_genie.agent] [ROUTE]     after router → career_coach
[resume_genie.agent] [NODE]      career_coach started


  [  0.3s] ROUTER → career_coach


[resume_genie.agent] [NODE]      career_coach completed (2293 chars, 1.77s)


  [  2.1s] TOOL: career_coach

Route: router → career_coach
Total time: 2.1s
Response length: 2293 chars

--- Response Preview (first 500 chars) ---
To determine a suitable salary range for your next role, I'll consider your background, experience, and skills as an Azure Big Data Engineer. Based on your resume, here are some key factors:

1. **Experience**: You have around 8-10 years of experience in software development, Big Data Analytics, and Azure Cloud Platform.
2. **Skills**: You possess a strong background in Big Data Analytics, ETL, Data Analytics, Azure Cloud Platform, Machine Learning, and Data Science.
3. **Certifications**: You h
...


In [13]:
print(msg.content)

To determine a suitable salary range for your next role, I'll consider your background, experience, and skills as an Azure Big Data Engineer. Based on your resume, here are some key factors:

1. **Experience**: You have around 8-10 years of experience in software development, Big Data Analytics, and Azure Cloud Platform.
2. **Skills**: You possess a strong background in Big Data Analytics, ETL, Data Analytics, Azure Cloud Platform, Machine Learning, and Data Science.
3. **Certifications**: You hold Microsoft Certified Azure Fundamentals, Microsoft Certified Azure Data Engineer, and other relevant certifications.
4. **Industry**: You have experience working in various industries, including banking, finance, and technology.
5. **Location**: Your location is not explicitly mentioned, but I'll assume you're based in India, given your phone numbers and email addresses.

Considering these factors, here are some salary ranges for Azure Big Data Engineer roles in India:

* **Average salary**: 

---
## 🔗 Part 6: Multi-Tool Chaining (with JD)

In [14]:
# ============================================================================
# TEST 5.1: Full Analysis — chains all 3 tools
# Expected: router → resume_checker → resume_scorer → cover_letter → responder
# ============================================================================
msg, _ = run_query("Give me a full analysis for this job", jd=job_description)


QUERY: Give me a full analysis for this job
JD: provided


[resume_genie.agent] [ROUTER]    query="Give me a full analysis for this job" | resume=uploaded | jd=provided
[resume_genie.agent] [ROUTER]    decision=multi|resume_checker,resume_scorer,cover_letter (0.30s)
[resume_genie.agent] [ROUTER]    tool_chain=['resume_checker', 'resume_scorer', 'cover_letter']
[resume_genie.agent] [ROUTE]     after router → resume_checker (multi, first in chain)
[resume_genie.agent] [NODE]      resume_checker started


  [  0.3s] ROUTER → multi chain=['resume_checker', 'resume_scorer', 'cover_letter']


[resume_genie.agent] [NODE]      resume_checker completed (2709 chars, 1.85s)
[resume_genie.agent] [ROUTE]     after tool → resume_scorer (next in chain, done={'resume_checker'})
[resume_genie.agent] [NODE]      resume_scorer started


  [  2.2s] TOOL: Resume Evaluation


[resume_genie.agent] [NODE]      resume_scorer completed (2658 chars, 1.78s)
[resume_genie.agent] [ROUTE]     after tool → cover_letter (next in chain, done={'resume_checker', 'resume_scorer'})
[resume_genie.agent] [NODE]      cover_letter started


  [  4.0s] TOOL: ATS Match Analysis


[resume_genie.agent] [NODE]      cover_letter completed (2925 chars, 2.04s)
[resume_genie.agent] [ROUTE]     after tool → responder (chain complete)
[resume_genie.agent] [RESPONDER] 3 tool output(s) → final message (8368 chars)


  [  6.0s] TOOL: Cover Letter
  [  6.0s] RESPONDER

Route: router → resume_checker → resume_scorer → cover_letter → responder
Total time: 6.0s
Response length: 8368 chars

--- Response Preview (first 500 chars) ---
## Resume Evaluation

**1. Score: 88/100**

The resume is well-structured, and the candidate has a strong background in Big Data Analytics, ETL, Data Analytics, and Azure Cloud Platform. However, there are some areas that need improvement, such as formatting and clarity in some sections.

**2. Strengths:**

1. **Strong technical skills**: The candidate has a wide range of technical skills, including Big Data Analytics, Machine Learning, Data Science, and Cloud Platforms.
2. **Experience in multi
...


In [15]:
# ============================================================================
# TEST 5.2: Full Analysis — alternate phrasing
# Expected: router → resume_checker → resume_scorer → cover_letter → responder
# ============================================================================
msg, _ = run_query("Prepare me completely for this job application", jd=job_description)

[resume_genie.agent] [ROUTER]    query="Prepare me completely for this job application" | resume=uploaded | jd=provided



QUERY: Prepare me completely for this job application
JD: provided


[resume_genie.agent] [ROUTER]    decision=multi|resume_checker,resume_scorer,cover_letter (0.16s)
[resume_genie.agent] [ROUTER]    tool_chain=['resume_checker', 'resume_scorer', 'cover_letter']
[resume_genie.agent] [ROUTE]     after router → resume_checker (multi, first in chain)
[resume_genie.agent] [NODE]      resume_checker started


  [  0.2s] ROUTER → multi chain=['resume_checker', 'resume_scorer', 'cover_letter']


[resume_genie.agent] [NODE]      resume_checker completed (2589 chars, 1.71s)
[resume_genie.agent] [ROUTE]     after tool → resume_scorer (next in chain, done={'resume_checker'})
[resume_genie.agent] [NODE]      resume_scorer started


  [  1.9s] TOOL: Resume Evaluation


[resume_genie.agent] [NODE]      resume_scorer completed (2873 chars, 1.88s)
[resume_genie.agent] [ROUTE]     after tool → cover_letter (next in chain, done={'resume_checker', 'resume_scorer'})
[resume_genie.agent] [NODE]      cover_letter started


  [  3.8s] TOOL: ATS Match Analysis


[resume_genie.agent] [NODE]      cover_letter completed (2136 chars, 1.44s)
[resume_genie.agent] [ROUTE]     after tool → responder (chain complete)
[resume_genie.agent] [RESPONDER] 3 tool output(s) → final message (7674 chars)


  [  5.3s] TOOL: Cover Letter
  [  5.3s] RESPONDER

Route: router → resume_checker → resume_scorer → cover_letter → responder
Total time: 5.3s
Response length: 7674 chars

--- Response Preview (first 500 chars) ---
## Resume Evaluation

**Score**: 88/100

**Strengths**:
1. **Comprehensive experience**: The candidate has extensive experience in Big Data Analytics, ETL, Data Analytics, and Azure Cloud Platform, with a proven track record of 8+ years in diverse facets of software development.
2. **Technical skills**: The candidate possesses a wide range of technical skills, including Big Data Analytics, Machine Learning, Data Science, and Cloud Platforms, with hands-on experience in managing Development Lifec
...


---
## 🔗 Part 7: Multi-Tool Chaining WITHOUT JD

In [16]:
# ============================================================================
# TEST 6.1: Full analysis without JD
# Expected: router → resume_checker → responder (JD tools filtered out)
# ============================================================================
msg, _ = run_query("Give me a full analysis", jd=None)

[resume_genie.agent] [ROUTER]    query="Give me a full analysis" | resume=uploaded | jd=NOT provided



QUERY: Give me a full analysis
JD: NOT provided


[resume_genie.agent] [ROUTER]    decision=multi|resume_checker,resume_scorer,cover_letter (0.26s)
[resume_genie.agent] [ROUTER]    filtered chain ['resume_checker', 'resume_scorer', 'cover_letter'] → ['resume_checker'] (no JD)
[resume_genie.agent] [ROUTER]    → single tool: resume_checker (from filtered multi)
[resume_genie.agent] [ROUTE]     after router → resume_checker
[resume_genie.agent] [NODE]      resume_checker started


  [  0.3s] ROUTER → resume_checker


[resume_genie.agent] [NODE]      resume_checker completed (3144 chars, 2.26s)
[resume_genie.agent] [ROUTE]     after tool → responder (single tool)
[resume_genie.agent] [RESPONDER] 1 tool output (resume_checker) → final message (3144 chars)


  [  2.5s] TOOL: Resume Evaluation
  [  2.5s] RESPONDER

Route: router → resume_checker → responder
Total time: 2.5s
Response length: 3144 chars

--- Response Preview (first 500 chars) ---
**1. Score: 85/100**

The resume is well-structured, and the candidate has a strong background in Big Data Analytics, Azure Cloud Platform, and Machine Learning. However, there are some areas for improvement, such as formatting and keyword optimization.

**2. Strengths:**

1. **Relevant experience**: The candidate has 8+ years of experience in diverse facets of software development, design, and execution of business applications, with a strong focus on Big Data Analytics and Cloud platforms.
2. 
...


---
## ⚠️ Part 8: Edge Cases — Missing JD

In [17]:
# ============================================================================
# TEST 7.1: Cover letter without JD
# Expected: router → END with "need JD" message
# ============================================================================
msg, _ = run_query("Write a cover letter", jd=None)

[resume_genie.agent] [ROUTER]    query="Write a cover letter" | resume=uploaded | jd=NOT provided



QUERY: Write a cover letter
JD: NOT provided


[resume_genie.agent] [ROUTER]    decision=cover_letter (0.26s)
[resume_genie.agent] [ROUTER]    → need_jd (LLM chose cover_letter but no JD provided — overriding)
[resume_genie.agent] [ROUTE]     after router → END (no tool needed)


  [  0.3s] ROUTER → none

Route: router
Total time: 0.3s
Response length: 117 chars

--- Response Preview (first 500 chars) ---
I'd need a job description to generate a cover letter. Please paste one in the sidebar or include it in your message.


In [18]:
# ============================================================================
# TEST 7.2: Score without JD
# Expected: router → END with "need JD" message
# ============================================================================
msg, _ = run_query("Score my resume against the JD", jd=None)

[resume_genie.agent] [ROUTER]    query="Score my resume against the JD" | resume=uploaded | jd=NOT provided



QUERY: Score my resume against the JD
JD: NOT provided


[resume_genie.agent] [ROUTER]    decision=need_jd (0.25s)
[resume_genie.agent] [ROUTER]    → need_jd (asking user for JD)
[resume_genie.agent] [ROUTE]     after router → END (no tool needed)


  [  0.3s] ROUTER → none

Route: router
Total time: 0.3s
Response length: 99 chars

--- Response Preview (first 500 chars) ---
I need a job description to do that. Please paste one in the sidebar or include it in your message.


---
## 🧠 Part 9: Conversation Memory

In [19]:
# ============================================================================
# TEST 8: Multi-turn conversation memory
# The follow-up should reference the first answer
# ============================================================================
print("--- Turn 1 ---")
msg1, history = run_query("What are my main weaknesses?")

# Add the AI response to history for the follow-up
if msg1:
    history.append(msg1)

print("\n\n--- Turn 2 (follow-up) ---")
msg2, _ = run_query("How do I fix those?", history=history)

[resume_genie.agent] [ROUTER]    query="What are my main weaknesses?" | resume=uploaded | jd=NOT provided


--- Turn 1 ---

QUERY: What are my main weaknesses?
JD: NOT provided


[resume_genie.agent] [ROUTER]    decision=resume_checker (0.35s)
[resume_genie.agent] [ROUTER]    → single tool: resume_checker
[resume_genie.agent] [ROUTE]     after router → resume_checker
[resume_genie.agent] [NODE]      resume_checker started


  [  0.4s] ROUTER → resume_checker


[resume_genie.agent] [NODE]      resume_checker completed (2557 chars, 1.80s)
[resume_genie.agent] [ROUTE]     after tool → responder (single tool)
[resume_genie.agent] [RESPONDER] 1 tool output (resume_checker) → final message (2557 chars)
[resume_genie.agent] [ROUTER]    query="How do I fix those?" | resume=uploaded | jd=NOT provided


  [  2.2s] TOOL: Resume Evaluation
  [  2.2s] RESPONDER

Route: router → resume_checker → responder
Total time: 2.2s
Response length: 2557 chars

--- Response Preview (first 500 chars) ---
**1. Score: 85/100**

The resume is well-structured, and the candidate has a strong background in Big Data Analytics, Cloud Platform, and Machine Learning. However, there are some areas for improvement, such as formatting, clarity, and relevance.

**2. Strengths:**

1. **Strong technical skills**: The candidate has a wide range of technical skills, including Big Data Analytics, Cloud Platform, Machine Learning, and Data Science.
2. **Experience in multiple projects**: The candidate has experienc
...


--- Turn 2 (follow-up) ---

QUERY: How do I fix those?
JD: NOT provided


[resume_genie.agent] [ROUTER]    decision=resume_checker (0.29s)
[resume_genie.agent] [ROUTER]    → single tool: resume_checker
[resume_genie.agent] [ROUTE]     after router → resume_checker
[resume_genie.agent] [NODE]      resume_checker started


  [  0.3s] ROUTER → resume_checker


[resume_genie.agent] [NODE]      resume_checker completed (3388 chars, 2.82s)
[resume_genie.agent] [ROUTE]     after tool → responder (single tool)
[resume_genie.agent] [RESPONDER] 1 tool output (resume_checker) → final message (3388 chars)


  [  3.1s] TOOL: Resume Evaluation
  [  3.1s] RESPONDER

Route: router → resume_checker → responder
Total time: 3.1s
Response length: 3388 chars

--- Response Preview (first 500 chars) ---
1. **Score**: 85 out of 100
2. **Strengths**:
   - The resume is comprehensive, covering a wide range of skills and experiences in Big Data Analytics, Cloud Computing, Machine Learning, and Data Science.
   - It showcases a strong background in handling various technologies and tools, including Azure, AWS, Hadoop, Spark, and numerous data processing and analytics tools.
   - The candidate has a proven track record of working in diverse roles, from developer to lead, across different industries a
...


In [20]:
print(msg2.content)

1. **Score**: 85 out of 100
2. **Strengths**:
   - The resume is comprehensive, covering a wide range of skills and experiences in Big Data Analytics, Cloud Computing, Machine Learning, and Data Science.
   - It showcases a strong background in handling various technologies and tools, including Azure, AWS, Hadoop, Spark, and numerous data processing and analytics tools.
   - The candidate has a proven track record of working in diverse roles, from developer to lead, across different industries and technologies, demonstrating versatility and adaptability.
   - The inclusion of certifications (e.g., Microsoft Certified Azure Fundamentals, Microsoft Certified Azure Data Engineer) adds credibility to the candidate's skills.
   - The resume provides specific details about projects undertaken, technologies used, and responsibilities handled, giving a clear picture of the candidate's capabilities.

3. **Weaknesses**:
   - The resume could benefit from a clearer structure and formatting to mak

---
## 🤷 Part 10: Ambiguous / General Queries

In [ ]:
# ============================================================================
# TEST 9.1: General greeting
# Expected: router → career_coach → END (should not crash)
# ============================================================================
msg, _ = run_query("Hello")

In [ ]:
# ============================================================================
# TEST 9.2: Capability question
# Expected: router → career_coach → END
# ============================================================================
msg, _ = run_query("What can you do?")

---
## 📝 Summary

| Category | Tests | Key Verification |
|----------|-------|-------------------|
| Resume Checker | 1.1, 1.2 | Routes without JD, returns score/strengths/weaknesses |
| Resume Scorer | 2.1, 2.2 | Routes with JD, returns ATS analysis |
| Cover Letter | 3.1 | Routes with JD, returns 300-450 word letter |
| Career Coach | 4.1, 4.2 | Routes for advice queries, conversational |
| Multi-Tool | 5.1, 5.2 | Chains 3 tools, combined output with section headers |
| Multi without JD | 6.1 | Filters JD-dependent tools, only runs checker |
| Missing JD | 7.1, 7.2 | Returns \"need JD\" message |
| Memory | 8 | Follow-up references prior context |
| Ambiguous | 9.1, 9.2 | Falls back to career coach, no crash |